# Homework 3 — Lead Scoring Classification
Target klasifikasi: `converted` (apakah client sign up ke platform atau tidak).

## Load Data
Read the 2026 lead-scoring dataset into a pandas DataFrame.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Check Missing Values
Check which columns contain missing values before filling them.

In [2]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

## Data Preparation
Fill missing values: categorical features with `'NA'`, numerical features with `0.0`.

In [3]:
categorical = list(df.select_dtypes(exclude='number').columns)  # kolom kategorikal (string)
numerical = [c for c in df.columns if c not in categorical and c != 'converted']

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

print("Categorical:", categorical)
print("Numerical:", numerical)
df.isnull().sum().sum()   # harus 0

Categorical: ['lead_source', 'industry', 'employment_status', 'location']
Numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


np.int64(0)

## Question 1
What is the most frequent observation (mode) for the column `industry`?

In [4]:
df['industry'].value_counts().head()

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64

In [5]:
q1 = df['industry'].mode()[0]
print("Q1 - mode industry:", q1)

Q1 - mode industry: technology


## Question 2
Correlation matrix for the numerical features. Only consider the four pairs listed in the homework.

In [6]:
corr = df[numerical].corr()
corr.round(3)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


In [7]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(f"{a} - {b}: {corr.loc[a, b]:.4f}")

q2 = max(pairs, key=lambda p: abs(corr.loc[p[0], p[1]]))
print("Q2 - biggest correlation:", q2)

interaction_count - lead_score: 0.9157
number_of_courses_viewed - lead_score: 0.7572
number_of_courses_viewed - interaction_count: 0.7216
annual_income - interaction_count: 0.1228
Q2 - biggest correlation: ('interaction_count', 'lead_score')


## Split the Data
60% train / 20% validation / 20% test, using the exact calls from the homework.

In [8]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# pastikan target tidak ikut jadi fitur
df_train = df_train.drop(columns='converted').reset_index(drop=True)
df_val = df_val.drop(columns='converted').reset_index(drop=True)
df_test = df_test.drop(columns='converted').reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Question 3
Mutual information between `converted` and each categorical variable (training set only), rounded to 2 decimals.

In [9]:
from sklearn.metrics import mutual_info_score

mi = {c: round(mutual_info_score(y_train, df_train[c]), 2) for c in categorical}
mi = pd.Series(mi).sort_values(ascending=False)
mi

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [10]:
# bandingkan nilai mentah juga, jaga-jaga kalau hasil pembulatan sama
raw_mi = pd.Series({c: mutual_info_score(y_train, df_train[c]) for c in categorical})
q3 = raw_mi.idxmax()
print("Q3 - biggest MI:", q3)

Q3 - biggest MI: lead_source


## Question 4
Logistic regression with one-hot encoding (`DictVectorizer`), accuracy on the validation set.

In [11]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


def train_and_eval(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [12]:
features = categorical + numerical
base_acc = train_and_eval(features)
q4 = round(base_acc, 2)
print("Accuracy:", base_acc)
print("Q4 - accuracy (2 desimal):", q4)

Accuracy: 0.645
Q4 - accuracy (2 desimal): 0.65


## Question 5
Feature elimination: drop each feature one at a time and compare to the base accuracy (unrounded).

In [13]:
diffs = {}
for f in features:
    subset = [x for x in features if x != f]
    acc = train_and_eval(subset)
    diffs[f] = base_acc - acc

diffs = pd.Series(diffs).sort_values()
diffs

annual_income              -0.079
industry                    0.000
location                    0.000
employment_status           0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

In [14]:
candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
# "smallest difference" = nilai selisih paling kecil (boleh negatif)
q5 = diffs[candidates].idxmin()
print(diffs[candidates])
print("Q5 - smallest difference:", q5)

lead_source                 0.003
number_of_courses_viewed    0.002
interaction_count           0.044
dtype: float64
Q5 - smallest difference: number_of_courses_viewed


## Question 6
Regularized logistic regression with different `C`, accuracy rounded to 3 decimals. If tied, pick the smallest `C`.

In [15]:
scores = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    scores[C] = round(train_and_eval(features, C=C), 3)
    print(f"C={C}: {scores[C]}")

best = max(scores.values())
q6 = min(C for C, s in scores.items() if s == best)   # ambil C terkecil kalau seri
print("Q6 - best C:", q6)

C=1e-06: 0.598


C=1e-05: 0.598


C=0.0001: 0.613


C=0.001: 0.645
Q6 - best C: 0.001


## Summary

In [16]:
pd.Series({
    'Q1': q1,
    'Q2': ' & '.join(q2),
    'Q3': q3,
    'Q4': q4,
    'Q5': q5,
    'Q6': q6,
}, name='answer')

Q1                        technology
Q2    interaction_count & lead_score
Q3                       lead_source
Q4                              0.65
Q5          number_of_courses_viewed
Q6                             0.001
Name: answer, dtype: object